# 02 用户分层与留存分析（RFM + Cohort）

> 基于订单事实表，完成：
> 1. 新老客结构拆解
> 2. RFM 用户分层与价值贡献分析
> 3. Cohort 留存分析（含成熟度标记）
>
> **口径说明：**
> - RFM 的 Recency 以「数据截止日 + 1 天」计算，确保最新购买用户 R=0
> - Cohort 仅横比 `m3_mature=True` 的成熟 cohort，尾部未成熟 cohort 不参与对比
> - RFM 分位采用 `rank(method='first')` 避免大量单次购买用户导致 qcut 边界重复

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'dashboard' / 'data'

fact = pd.read_csv(DATA_DIR / 'order_fact.csv',
                   parse_dates=['order_purchase_timestamp'])
valid = fact[fact['is_valid_analysis']].copy()
print(f"有效订单: {len(valid):,} | 用户: {valid['customer_unique_id'].nunique():,}")

## 1. 新老客结构拆解

识别每个用户的首购月份，判断当月是否为新客。

In [ ]:
# 识别首购
valid['first_purchase'] = valid.groupby('customer_unique_id')['order_purchase_timestamp'].transform('min')
valid['is_new'] = (valid['order_purchase_timestamp'].dt.to_period('M') ==
                   valid['first_purchase'].dt.to_period('M'))

# 月度新老客
monthly_new = valid.groupby('purchase_month').agg(
    total_users=('customer_unique_id', 'nunique'),
    new_users=('customer_unique_id', lambda x: x[valid.loc[x.index, 'is_new']].nunique()),
).reset_index()
monthly_new['repeat_users'] = monthly_new['total_users'] - monthly_new['new_users']
monthly_new['new_ratio'] = monthly_new['new_users'] / monthly_new['total_users']

# 排除不完整月（最后一个月）
monthly_new = monthly_new.iloc[:-1]

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(monthly_new['purchase_month'], monthly_new['new_users'], label='新客', color='steelblue')
ax.bar(monthly_new['purchase_month'], monthly_new['repeat_users'],
       bottom=monthly_new['new_users'], label='老客', color='orange')
ax.set_title('月度新老客结构')
ax.set_ylabel('用户数')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"整体新客占比: {monthly_new['new_users'].sum() / monthly_new['total_users'].sum():.1%}")
print(f"观察窗复购率: {(valid.groupby('customer_unique_id')['order_id'].nunique() >= 2).mean():.1%}")

## 2. RFM 用户分层

**RFM 定义：**
- **R (Recency)**：最近一次购买距数据截止日+1的天数，越小越好
- **F (Frequency)**：观察窗内订单数
- **M (Monetary)**：观察窗内 GMV 总和

每个维度按五分位打分 1-5，R 越小分越高，F/M 越大分越高。总分 3-15，分段：
- 高价值（≥12）、潜力（9-11）、需唤回（6-8）、沉睡（<6）

In [ ]:
rfm = pd.read_csv(DATA_DIR / 'rfm_customers.csv')
print(f"RFM 分层用户总数: {len(rfm):,}")
print()
print("各维度统计:")
print(rfm[['recency_days', 'frequency', 'monetary']].describe().to_string())
print()
print("分段分布:")
print(rfm['segment'].value_counts().to_string())

In [ ]:
# 各分层价值贡献
rfm_summary = pd.read_csv(DATA_DIR / 'rfm_summary.csv')
rfm_summary['revenue_pct'] = rfm_summary['revenue'] / rfm_summary['revenue'].sum()
rfm_summary['user_pct'] = rfm_summary['users'] / rfm_summary['users'].sum()
print(rfm_summary[['segment', 'users', 'user_pct', 'revenue', 'revenue_pct', 'avg_frequency']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(rfm_summary['users'], labels=rfm_summary['segment'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('用户数占比')
axes[1].pie(rfm_summary['revenue'], labels=rfm_summary['segment'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('GMV 贡献占比')
plt.tight_layout()
plt.show()

high = rfm_summary[rfm_summary['segment'] == '高价值'].iloc[0]
print(f"高价值用户: {high['users']:,} 人 ({high['user_pct']:.1%})，贡献 GMV R$ {high['revenue']:,.0f} ({high['revenue_pct']:.1%})")
print(f"高价值组平均频次仅 {high['avg_frequency']:.2f} 单——二单转化空间显著。")

In [ ]:
# RFM 散点：F vs M，颜色按分段
fig, ax = plt.subplots(figsize=(10, 6))
for seg in ['高价值', '潜力', '需唤回', '沉睡']:
    sub = rfm[rfm['segment'] == seg]
    ax.scatter(sub['frequency'], sub['monetary'], label=seg, alpha=0.5, s=20)
ax.set_xlabel('Frequency (订单数)')
ax.set_ylabel('Monetary (GMV R$)')
ax.set_title('RFM 分层散点图')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 3. Cohort 留存分析

按用户首购月份分 cohort，跟踪后续各月的留存率。
**关键处理：**
- 补齐 0 活跃月份，区分「0 活跃」与「无数据」
- 标记 `m3_mature`（观察月数 ≥ 3），仅横比成熟 cohort

In [ ]:
cohort = pd.read_csv(DATA_DIR / 'cohort_retention.csv')
print(f"Cohort 记录数: {len(cohort):,}")
print(f"首购月份范围: {cohort['cohort_month'].min()} ~ {cohort['cohort_month'].max()}")
print(f"成熟 cohort (m3_mature=True): {cohort[cohort['m3_mature']==True]['cohort_month'].nunique()} 个")
print()
print("各 cohort 首月用户数:")
cohort_size = cohort[cohort['period'] == 0][['cohort_month', 'cohort_size']]
print(cohort_size.to_string(index=False))

In [ ]:
# 留存热力图（仅展示成熟 cohort）
mature = cohort[cohort['m3_mature'] == True].copy()
pivot = mature.pivot(index='cohort_month', columns='period', values='retention_rate')

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, annot=True, fmt='.1%', cmap='YlOrRd', ax=ax, cbar_kws={'label': '留存率'})
ax.set_title('Cohort 留存热力图（仅成熟 cohort，m3_mature=True）')
ax.set_xlabel('首购后第 N 月')
ax.set_ylabel('首购月份 cohort')
plt.tight_layout()
plt.show()

# 关键留存指标
print("各 cohort M1 留存率:")
m1 = mature[mature['period'] == 1][['cohort_month', 'retention_rate']]
print(m1.to_string(index=False))
print(f"平均 M1 留存率: {m1['retention_rate'].mean():.1%}")

## 4. 小结与策略含义

| 发现 | 策略含义 |
|---|---|
| 复购率仅 3.0%，高价值组平均频次 1.14 | 二单转化是最大增长杠杆，P1 高价值用户关怀 |
| 高价值用户占比 ~16.6%，贡献 ~29.7% GMV | 优先保护高价值人群，权益投入 ROI 最高 |
| 需唤回人群规模大 | P2 小流量实验验证触达策略，防自然购买前置 |
| M1 留存率极低 | 新客首单体验（履约、商品）是留存基础，呼应 P0 履约治理 |

> 下一步：模拟 A/B 实验设计与统计检验（见 `03_ab_test.ipynb`）